In [1]:
import os
import copy
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

In [2]:
BASE_DIR = r"C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR = f"{BASE_DIR}/classification/val"
TEST_DIR = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("SETUP COMPLETE")
print("Device:", device)

SETUP COMPLETE
Device: cuda


In [3]:
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS = 15
NUM_CLASSES = 26

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomRotation(20),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.85, 1.15)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH,
    shuffle=False
)

CLASS_NAMES = train_dataset.classes

print(f"\nNUMBER OF CLASSES: {len(CLASS_NAMES)}")
print(CLASS_NAMES)


NUMBER OF CLASSES: 26
['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'train', 'truck']


In [4]:
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

print("\nPRETRAINED RESNET50 LOADED")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\SAKTHI/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:30<00:00, 3.35MB/s]



PRETRAINED RESNET50 LOADED


In [5]:
for parameter in model.parameters():
    parameter.requires_grad = False

In [6]:
input_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.BatchNorm1d(input_features),
    nn.Dropout(0.5),
    nn.Linear(input_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES)
)

In [7]:
model = model.to(device)

In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = Adam(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=1e-4
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

print("\nMODEL COMPILED")


MODEL COMPILED


In [9]:
BEST_MODEL_PATH = "models/ResNet50_best.pth"
PATIENCE = 5

In [10]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [11]:
print("\nSTARTING INITIAL TRAINING")

history = {
    "accuracy": [],
    "top5_accuracy": [],
    "val_accuracy": [],
    "val_top5_accuracy": [],
    "loss": [],
    "val_loss": []
}

best_val_accuracy = 0.0
epochs_without_improvement = 0
best_model_state = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_top5_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_top5_correct += (
            outputs.topk(5, dim=1).indices == labels.unsqueeze(1)
        ).any(dim=1).sum().item()
        train_total += labels.size(0)

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_top5_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_top5_correct += (
                outputs.topk(5, dim=1).indices == labels.unsqueeze(1)
            ).any(dim=1).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total
    train_top5_accuracy = train_top5_correct / train_total

    val_loss /= val_total
    val_accuracy = val_correct / val_total
    val_top5_accuracy = val_top5_correct / val_total

    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)
    history["top5_accuracy"].append(train_top5_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["val_top5_accuracy"].append(val_top5_accuracy)

    scheduler.step(val_loss)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} "
        f"Accuracy: {train_accuracy:.4f} "
        f"Top-5: {train_top5_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Accuracy: {val_accuracy:.4f} "
        f"Val Top-5: {val_top5_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, BEST_MODEL_PATH)
        print("Best model saved.")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(best_model_state)


STARTING INITIAL TRAINING
Epoch [1/15] Loss: 3.2190 Accuracy: 0.0670 Top-5: 0.2890 Val Loss: 3.1349 Val Accuracy: 0.1692 Val Top-5: 0.5051
Best model saved.
Epoch [2/15] Loss: 2.9841 Accuracy: 0.2181 Top-5: 0.5110 Val Loss: 2.9710 Val Accuracy: 0.3154 Val Top-5: 0.5769
Best model saved.
Epoch [3/15] Loss: 2.7668 Accuracy: 0.3385 Top-5: 0.6604 Val Loss: 2.8179 Val Accuracy: 0.3667 Val Top-5: 0.6744
Best model saved.
Epoch [4/15] Loss: 2.5850 Accuracy: 0.3929 Top-5: 0.7154 Val Loss: 2.6878 Val Accuracy: 0.3846 Val Top-5: 0.7128
Best model saved.
Epoch [5/15] Loss: 2.4271 Accuracy: 0.4451 Top-5: 0.7637 Val Loss: 2.5387 Val Accuracy: 0.4615 Val Top-5: 0.7513
Best model saved.
Epoch [6/15] Loss: 2.2930 Accuracy: 0.4973 Top-5: 0.7929 Val Loss: 2.4490 Val Accuracy: 0.4872 Val Top-5: 0.7615
Best model saved.
Epoch [7/15] Loss: 2.1875 Accuracy: 0.5187 Top-5: 0.8170 Val Loss: 2.3868 Val Accuracy: 0.4359 Val Top-5: 0.7744
Epoch [8/15] Loss: 2.1342 Accuracy: 0.5302 Top-5: 0.8220 Val Loss: 2.3079 

<All keys matched successfully>